In [1]:
import numpy as np
import nibabel as nb
import Functional_Fusion.dataset as fdata
import cortico_cereb_connectivity.globals as gl
import cortico_cereb_connectivity.summarize as cs
import cortico_cereb_connectivity.scripts.script_plotting_utils as plu
import matplotlib.pyplot as plt

In [ ]:
traindata = gl.traindata_string(None)
cortex_roi = "Icosahedron1002"
cerebellum_roi = "NettekovenSym32"
# method = 'L2reg';       ex = 'A4_global'
method = 'NNLS';        ex = 'A1_global'

fname = gl.conn_dir + f'/maps/{traindata}_{cortex_roi}_{cerebellum_roi}_{method}_mean.pscalar.nii'
oname = f'{traindata}_{cortex_roi}_{cerebellum_roi}_{method}_mean.dscalar.nii'

In [9]:
cifti_img = cs.avrg_weight_map_roi(traindata=traindata,
                                   cortex_roi=cortex_roi,
                                   method=method,
                                   extension=ex,
                                   cerebellum_roi=cerebellum_roi,
                                   norm=True,
                                   ynorm=True)
cifti_img_new = cs.sort_roi_rows(cifti_img)
nb.save(cifti_img_new, fname)

In [10]:
cifti_img_new = nb.load(fname)
labels = cifti_img_new.header.get_axis(0).name
# index,colors,labels = nt.read_lut(gl.atlas_dir + f"/tpl-MNI152NLin2009cSymC/atl-{cerebellum_roi}.lut")

In [11]:
# smooth the cifti
cs.pscalar_to_smoothed_dscalar(cifti_img_new, oname, sigma=2.0)

wb_command -cifti-smoothing 'temp.dscalar.nii' 2.0 0.0 COLUMN /cifs/diedrichsen/data/Cerebellum/connectivity/maps/MdWfIbDeHtNiSoScLa_Icosahedron1002_NettekovenSym32_NNLS_mean.dscalar.nii -left-surface /cifs/diedrichsen/data/FunctionalFusion_new/Atlases/tpl-fs32k/tpl-fs32k_hemi-L_inflated.surf.gii -right-surface /cifs/diedrichsen/data/FunctionalFusion_new/Atlases/tpl-fs32k/tpl-fs32k_hemi-R_inflated.surf.gii


In [10]:
def make_cifti_subset(cifti_img, region_index):
    data = cifti_img.get_fdata()
    
    # Keep the selected row as (1, n_brainordinates)
    row = data[region_index:region_index + 1, :]

    # Get the two CIFTI axes
    axis0 = cifti_img.header.get_axis(0)
    axis1 = cifti_img.header.get_axis(1)

    # Subset only the first axis
    new_axis0 = axis0[region_index:region_index + 1]

    # Construct a header with the correct axes
    header = nb.cifti2.Cifti2Header.from_axes(
        (new_axis0, axis1)
    )

    img = nb.Cifti2Image(
        row,
        header=header,
        nifti_header=cifti_img.nifti_header
    )

    return img

In [11]:
for i, label in enumerate(labels):
    print(f"Label {i+1}: {label}")
    cifti_img_region = make_cifti_subset(cifti_img_new, i)
    # make the smoothed version of that map
    oname = f'{traindata}_{cortex_roi}_{label}_{method}_mean.dscalar.nii'
    cs.pscalar_to_smoothed_dscalar(cifti_img_region, oname, sigma=2.0)

Label 1: 0
wb_command -cifti-smoothing 'temp.dscalar.nii' 2.0 0.0 COLUMN /cifs/diedrichsen/data/Cerebellum/connectivity/maps/MdWfIbDeHtNiSoScLa_Icosahedron1002_0_L2reg_mean.dscalar.nii -left-surface /cifs/diedrichsen/data/FunctionalFusion_new/Atlases/tpl-fs32k/tpl-fs32k_hemi-L_inflated.surf.gii -right-surface /cifs/diedrichsen/data/FunctionalFusion_new/Atlases/tpl-fs32k/tpl-fs32k_hemi-R_inflated.surf.gii
Label 2: 1
wb_command -cifti-smoothing 'temp.dscalar.nii' 2.0 0.0 COLUMN /cifs/diedrichsen/data/Cerebellum/connectivity/maps/MdWfIbDeHtNiSoScLa_Icosahedron1002_1_L2reg_mean.dscalar.nii -left-surface /cifs/diedrichsen/data/FunctionalFusion_new/Atlases/tpl-fs32k/tpl-fs32k_hemi-L_inflated.surf.gii -right-surface /cifs/diedrichsen/data/FunctionalFusion_new/Atlases/tpl-fs32k/tpl-fs32k_hemi-R_inflated.surf.gii
Label 3: 10
wb_command -cifti-smoothing 'temp.dscalar.nii' 2.0 0.0 COLUMN /cifs/diedrichsen/data/Cerebellum/connectivity/maps/MdWfIbDeHtNiSoScLa_Icosahedron1002_10_L2reg_mean.dscalar.n